In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [6]:
print("="*60)
print("SALES PREDICTION USING MACHINE LEARNING")
print("="*60)

np.random.seed(42)
n_samples = 200

SALES PREDICTION USING MACHINE LEARNING


In [10]:
df=pd.read_csv('Advertising.csv')
df.head()

,Unnamed: 0,TV,Radio,Newspaper,Sales
0,1,230.1,37.8,69.2,22.1
1,2,44.5,39.3,45.1,10.4
2,3,17.2,45.9,69.3,9.3
3,4,151.5,41.3,58.5,18.5
4,5,180.8,10.8,58.4,12.9


In [8]:
df['Sales'] = (
    5 + 
    0.05 * df['TV'] + 
    0.15 * df['Radio'] + 
    0.03 * df['Newspaper'] + 
    np.random.normal(0, 2, n_samples)
)


In [ ]:
if 'Platform' not in df.columns:
    df['Platform'] = np.random.choice(['Digital', 'Traditional', 'Social Media'], size=len(df), p=[0.4, 0.35, 0.25])

if 'Target_Segment' not in df.columns:
    df['Target_Segment'] = np.random.choice(['Youth', 'Adults', 'Seniors'], size=len(df), p=[0.3, 0.5, 0.2])

platform_effect = {'Digital': 1.2, 'Traditional': 1.0, 'Social Media': 1.3}
segment_effect = {'Youth': 1.1, 'Adults': 1.0, 'Seniors': 0.9}

df['Sales'] = (
    df['Sales']
    * df['Platform'].map(platform_effect).fillna(1.0)
    * df['Target_Segment'].map(segment_effect).fillna(1.0)
)

KeyError: 'Platform'

In [ ]:
print("\n1. DATASET OVERVIEW")
print("-"*60)
print(f"Total Records: {len(df)}")
print(f"Columns: {list(df.columns)}")
print("\nFirst 5 Rows:")
print(df.head())
print("\nStatistical Summary:")
print(df.describe())

print("\n2. DATA VISUALIZATION")
print("-"*60)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

sns.scatterplot(data=df, x='TV', y='Sales', ax=axes[0, 0], color='blue', alpha=0.6)
axes[0, 0].set_title('TV Advertising vs Sales')

sns.scatterplot(data=df, x='Radio', y='Sales', ax=axes[0, 1], color='green', alpha=0.6)
axes[0, 1].set_title('Radio Advertising vs Sales')

sns.scatterplot(data=df, x='Newspaper', y='Sales', ax=axes[0, 2], color='red', alpha=0.6)
axes[0, 2].set_title('Newspaper Advertising vs Sales')

sns.boxplot(data=df, x='Platform', y='Sales', ax=axes[1, 0])
axes[1, 0].set_title('Sales by Platform')
axes[1, 0].tick_params(axis='x', rotation=45)

sns.boxplot(data=df, x='Target_Segment', y='Sales', ax=axes[1, 1])
axes[1, 1].set_title('Sales by Target Segment')
axes[1, 1].tick_params(axis='x', rotation=45)

sns.histplot(data=df, x='Sales', kde=True, ax=axes[1, 2], color='purple')
axes[1, 2].set_title('Distribution of Sales')

plt.tight_layout()
plt.savefig('sales_analysis.png', dpi=300)
print("Charts saved as 'sales_analysis.png'")

In [ ]:
df_encoded = pd.get_dummies(df, columns=['Platform', 'Target_Segment'], drop_first=True)

X = df_encoded.drop('Sales', axis=1)
y = df_encoded['Sales']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
print("\n3. MODEL TRAINING & COMPARISON")
print("-"*60)

models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42)
}

In [ ]:
results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    r2 = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    
    results[name] = {
        'model': model,
        'r2': r2,
        'rmse': rmse,
        'mae': mae,
        'predictions': y_pred
    }

In [ ]:
print(f"\n{name}:")
print(f"  R² Score: {r2:.4f}")
print(f"  RMSE: {rmse:.4f}")
print(f"  MAE: {mae:.4f}")

best_model_name = max(results, key=lambda x: results[x]['r2'])
best_model = results[best_model_name]['model']

print(f"\n✓ BEST MODEL: {best_model_name}")

In [ ]:
print("\n4. ADVERTISING IMPACT ANALYSIS")
print("-"*60)

if hasattr(best_model, 'feature_importances_'):
    importances = pd.DataFrame({
        'Feature': X.columns,
        'Importance': best_model.feature_importances_
    }).sort_values('Importance', ascending=False)
    
    print("\nFeature Importance (Top 5):")
    print(importances.head())
    
    plt.figure(figsize=(10, 6))
    sns.barplot(data=importances.head(10), x='Importance', y='Feature')
    plt.title('Feature Importance - ' + best_model_name)
    plt.tight_layout()
    plt.savefig('feature_importance.png', dpi=300)
    print("Feature importance chart saved!")

In [ ]:
print("\n5. BUSINESS SCENARIOS & PREDICTIONS")
print("-"*60)

avg_tv = df['TV'].mean()
avg_radio = df['Radio'].mean()
avg_newspaper = df['Newspaper'].mean()

baseline = pd.DataFrame({
    'TV': [avg_tv],
    'Radio': [avg_radio],
    'Newspaper': [avg_newspaper],
    'Platform_Traditional': [0],
    'Platform_Social Media': [0],
    'Target_Segment_Adults': [0],
    'Target_Segment_Seniors': [0]
})

In [ ]:
for col in X.columns:
    if col not in baseline.columns:
        baseline[col] = 0

baseline = baseline[X.columns]

baseline_sales = best_model.predict(baseline)[0]
print(f"\nBaseline Scenario (Average Spending):")
print(f"  TV: ${avg_tv:.0f}, Radio: ${avg_radio:.0f}, Newspaper: ${avg_newspaper:.0f}")
print(f"  Predicted Sales: ${baseline_sales:.2f}")

scenario_1 = baseline.copy()
scenario_1['TV'] = avg_tv * 1.5
sales_1 = best_model.predict(scenario_1)[0]
print(f"\nScenario 1 - Increase TV by 50%:")
print(f"  Predicted Sales: ${sales_1:.2f}")
print(f"  Improvement: {((sales_1/baseline_sales)-1)*100:.1f}%")

scenario_2 = baseline.copy()
scenario_2['Radio'] = avg_radio * 1.5
sales_2 = best_model.predict(scenario_2)[0]
print(f"\nScenario 2 - Increase Radio by 50%:")
print(f"  Predicted Sales: ${sales_2:.2f}")
print(f"  Improvement: {((sales_2/baseline_sales)-1)*100:.1f}%")

scenario_3 = baseline.copy()
scenario_3['TV'] = avg_tv * 1.3
scenario_3['Radio'] = avg_radio * 1.3
sales_3 = best_model.predict(scenario_3)[0]
print(f"\nScenario 3 - Increase TV & Radio by 30%:")
print(f"  Predicted Sales: ${sales_3:.2f}")
print(f"  Improvement: {((sales_3/baseline_sales)-1)*100:.1f}%")

In [ ]:
print("\n6. MODEL PERFORMANCE VISUALIZATION")
print("-"*60)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

model_names = list(results.keys())
r2_scores = [results[m]['r2'] for m in model_names]

axes[0].bar(model_names, r2_scores, color='skyblue')
axes[0].set_ylabel('R² Score')
axes[0].set_title('Model Comparison - R² Score')
axes[0].tick_params(axis='x', rotation=45)

best_predictions = results[best_model_name]['predictions']
axes[1].scatter(y_test, best_predictions, alpha=0.6, color='green')
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
axes[1].set_xlabel('Actual Sales')
axes[1].set_ylabel('Predicted Sales')
axes[1].set_title(f'{best_model_name}\nActual vs Predicted')

errors = np.abs(y_test.values - best_predictions)
axes[2].hist(errors, bins=20, color='orange', edgecolor='black')
axes[2].set_xlabel('Prediction Error')
axes[2].set_ylabel('Frequency')
axes[2].set_title('Distribution of Errors')

plt.tight_layout()
plt.savefig('model_performance.png', dpi=300)